# **TAS (Tele Assistance System) Dimensional Analysis Modelling**
NOTE: _[DASA CASE STUDY 1]_

## **Summary**

This notebook is focused on three main objectives:
1. summarizing the key aspects of the Tele Assistance System (TAS) architecture and its adaptive capabilities in the context of telehealth services for chronic patients.
2. Modelling the TAS architecture using appropriate design notations and tools to visualize its components and interactions.
3. Simulating the TAS behavior under different scenarios to evaluate its performance and adaptability with Queue Network (QN) models.

The results will be used to evaluate the Dimensional Analysis for CUSTOM Architecture (DASA) methodology, its CUSTOM tool (PyDASA) and its effectiveness in modelling and Quality Scenarios (QS) trade-off in self-adaptive-systems (SAS).

---

## **Software Architecture**
- TAS (Tele Assistance System) operates in a dynamic environment where service quality, availability, and user needs frequently change.
- The TAS is further subdivided into Controller and Target System subsystem components.
- The Controller is responsible for managing the overall system behavior, while the Target System focuses on executing specific tasks related to patient care.
- The TAS target systems follows a Service-oriented architecture (SOA) pattern.
- The TAS Controller follows a MAPE-K (Monitor-Analyze-Plan-Execute-Knowledge) feedback loop for self-adaptation.
- Adaptations focus on maintaining **reliability**, **performance**, and **compliance** with patient care standards (5 specific scenarios).
- ActivFORMS provides the runtime framework for model-based adaptation using runtime models, simulations, and verified decision-making.

---

_**NOTE: MORE DETAILS ON THE ARCHITECTURE IN THE ANALYTICAL MODELLING NOTEBOOK!.**_

---

## **Code**

_**SUMMARY:**_

This code is for the analytical solution of the Case Study (TAS) Dimensional Analysis Model and is structured as follows:
1. Analytical Dimensional Model (DA).
2. Importing necessary libraries and modules.
3. Loading DA default configuration.
4. Solving the DA analytically.
5. Solving the DA with Monte Carlo simulations.
6. Plotting the DA with the obtained metrics.
7. Loading DA 'optimal' configuration.
8. Solving the DA optimally.
9. Simulating the DA with the 'optimal' configuration.
10. Plotting the optimal DA with the obtained metrics.
11. Saving the results.
12. Comparing the analytical results (Default Vs. Optimal)
13. Visualizing the results.
14. Generating a summary report.

## **Target System Queue Network Model**

<svg viewBox="0 0 4650 2000" width="1400" height="650">
    <!-- SVG content -->
    <image href="assets/cs1/img/04A - Queue Network.svg" alt="queue-net-diagram" />
    <div align="center"><em>Image 4. TAS Queue Network Diagram.</em></div>
</svg>

### **Necessary Imports**

In [1]:
# -*- coding: utf-8 -*-
# Native imports
import os
import re
import sys
import time
from typing import Union

# Third-party imports, data and numeric
import numpy as np
import pandas as pd

# Model Aprox + Visualizartion libraries
from scipy.interpolate import Rbf
from scipy.interpolate import griddata, LinearNDInterpolator
# from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.ensemble import GradientBoostingRegressor

import matplotlib.pyplot as plt

# import qimensional model libraries PyDASA
# config module
from pydasa.utils import config
# FDU modules
from pydasa.core.fundamental import Dimension
# FDU regex management
from pydasa.dimensional.framework import DimScheme
# Variable and Variable modules
from pydasa.core.parameter import Variable
# Dimensional Matrix Modelling module
from pydasa.dimensional.model import DimMatrix
# sensitivity analysis modules
from pydasa.analysis.scenario import DimSensitivity
from pydasa.handlers.influence import SensitivityHandler
# Monte Carlo Simulation modules
from pydasa.analysis.simulation import MonteCarloSim
from pydasa.handlers.practical import MonteCarloHandler

# import queue network + models packages
from src.model.queueing import Queue
from src.model.analytical import solve_jackson_network, calculate_net_metrics

# import plot functions + grahics
from src.view.plots import plot_queue_network
from src.view.plots import plot_net_comparison
from src.view.plots import plot_net_difference
from src.view.plots import plot_nodes_heatmap
from src.view.plots import plot_nodes_diffmap
from src.view.plots import plot_performance_coef_chart
from src.view.plots import plot_experiment_coef_chart

### **Function Definitions**

In [ ]:
# Simple formatter for console output

def fmt(val: Union[int, float, np.number]) -> Union[str, np.ndarray]:
    """Format a number to 4 decimal places for console output.

    Args:
        val (Union[int, float, np.number, np.ndarray]): The value to format.

    Returns:
        Union[str, np.ndarray]: The formatted value as a string or an array of strings.
    """
    if isinstance(val, (int, float, np.number)):
        if np.isnan(val) or np.isinf(val):
            return str(val)
        return f"{val:.4f}"
    elif isinstance(val, np.ndarray):
        return np.array([fmt(x) for x in val])
    return val

In [ ]:
# Load configuration from a CSV file
def load(path: str, fname: str) -> pd.DataFrame:
    """Load configuration from a CSV file.

    Args:
        path (str): The directory path where the CSV file is located.
        fname (str): The name of the CSV file to load.

    Returns:
        pd.DataFrame: A DataFrame containing the configuration data.
            CSV format:
                - node: <node_id>
                - miu: <mean_service_time>
                - c: <service_channels>
                - K: <buffer_capacity | max_queue_length>
                - lambda0: <initial_arrival_rate>
                - L0: <initial_queue_length>
                - pm: <matrix_routing_probabilities>
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Loading configuration from: {_file_path}")
    df = pd.read_csv(_file_path)
    return df

In [ ]:
# save dataframes in CSV files
def save(path: str, fname: str, data: pd.DataFrame) -> None:
    """Save a DataFrame to a CSV file.

    Args:
        path (str): The directory path where the CSV file will be saved.
        fname (str): The name of the CSV file to save.
        data (pd.DataFrame): The DataFrame containing the data to save.
    """
    # path = os.path.dirname(__file__)
    _file_path = os.path.join(path, fname)
    print(f"Saving data to: {_file_path}")
    data.to_csv(_file_path, index=False)

In [ ]:
# calculate queue length, with ququeing theory formulas
def queue_length(lambda_z: float,
                 mu: float,
                 c: int,
                 K: int) -> float:
    """Calculate the average queue length (Lq) for an M/M/c/K queueing system.

    Args:
        lambda_z (float): The arrival rate (λ).
        mu (float): The service rate (μ).
        c (int): The number of service channels.
        K (int): The maximum number of customers in the system.

    Returns:
        float: The average number of requests in the system (L, Lq).
    """
    adjusted = False
    # to avoid unstable system
    if lambda_z > mu * c:
        lambda_z = mu * 0.9999
        adjusted = True

    # create queue object
    q = Queue("M/M/s/K", lambda_z, mu, c, K)
    # calculate queue metrics
    q.calculate_metrics()
    ans = q.avg_len

    # to know afterwars what data is adjusted in df
    if adjusted is True:
        # ans = -1 * ans
        ans = (lambda_z, mu, c, K, ans)

    # check the results in the dataframe afterwars
    return ans

In [ ]:
# calculate queue time, with Little's Law
def queue_time(lambda_z: float, len_queue: float) -> float:
    """Calculate the average time in the queue system (W, Wq).

    Args:
        len_queue (float): The average queue length (L, Lq).
        lambda_z (float): The arrival rate (λ).

    Returns:
        float: The average wait time in the queue (W, Wq).
    """
    if lambda_z == 0:
        return float("inf")  # Avoid division by zero
    wait_time = len_queue / lambda_z
    return wait_time

In [ ]:
def adjust_rate(rate:float, err:float=0.0) -> float:
    """Adjust the arrival rate by a given error percentage.

    Args:
        rate (float): The original arrival rate (λ).
        err (float): The error percentage to adjust the rate by.

    Returns:
        float: The adjusted arrival rate otherwise known as departure rate (x).
    """
    adjusted_rate = rate * (1 + err)
    return adjusted_rate

In [ ]:
def uniform_int(low: int, high: int) -> int:
    """Generate a random integer within a specified range.

    Args:
        low (int): The lower bound of the range (inclusive).
        high (int): The upper bound of the range (exclusive).

    Returns:
        int: A random integer between low and high.
    """
    return np.random.randint(low, high)

In [ ]:
def shifted_uniform_int(low: int, high: int, shift: int) -> int:
    """Generate a random integer within a specified range and apply a shift.

    Args:
        low (int): The lower bound of the range (inclusive).
        high (int): The upper bound of the range (exclusive).
        shift (int): The value to shift the generated number by.

    Returns:
        int: A random integer between low and high, shifted by the specified amount.
    """
    return np.random.randint(low, high) + shift

In [ ]:
def uniform(low: float, high: float) -> float:
    """Generate a random float within a specified range.

    Args:
        low (float): The lower bound of the range (inclusive).
        high (float): The upper bound of the range (exclusive).

    Returns:
        float: A random float between low and high.
    """
    return np.random.uniform(low, high)

In [ ]:
def create_dimensional_groups(config: pd.DataFrame,
                              verbose: bool = False) -> dict:
    """Create dimensional variable groups from the default configuration.

    Args:
        config (pd.DataFrame): The configuration DataFrame.
        verbose (bool, optional): Whether to print detailed information. Defaults to False.

    Returns:
        dict: A dictionary containing groups of dimensional variables.
    """
    # create a dimensional set of variables from config file
    # dimensional groups dictionary
    dim_groups = {}
    
    # unique dimensional matrix
    unique_dim_groups = config["dm"].unique().tolist()
    for group in unique_dim_groups:
        _tdict = dict()
        # filter by dimensional matrix number
        _relevance_lt = config[config["dm"] == group]
        # filter by relevant attributes
        _relevance_lt = _relevance_lt[_relevance_lt["dm"] == group]
        if verbose:
            _msg = f"Dimensional Matrix: {group}, "
            _msg += f"with {_relevance_lt.shape[0]} relevant vars."
            print(_msg)

        # create variable instances
        for var in _relevance_lt.to_dict(orient="records"):
            key = var["_sym"]
            # remove unnecessary keys
            var.pop("dm", None)

            # cast internal dict column for dist_params
            if var.get("_dist_params") != None:
                data = eval(var.get("_dist_params"))
                dt = {"_dist_params": data}
                var.update(dt)

            # cast internal dict column for dependences
            if var.get("_depends") != None:
                data = eval(var.get("_depends"))
                dt = {"_depends": data}
                var.update(dt)
            # create variable instance
            _tdict[key] = Variable(**var)
            if verbose:
                _msg = f"\tCreated Variable Instance: {key}: {_tdict[key]}"
                print(_msg)

        # add to dictionary
        dim_groups[group] = _tdict

    return dim_groups

In [ ]:
def configure_dimensional_dist(dim_groups: dict,
                               verbose: bool = False) -> dict:
    """Configure dimensional variable distributions for a specific group.

    Args:
        dim_groups (dict): A dictionary containing the groups of dimensional variables.
        verbose (bool, optional): Whether to print detailed information. Defaults to False.

    Returns:
        dict: A dictionary containing the dimensional group with configured distribution functions in its variables.
    """

    # select the dimensional group to configure
    # im tired so lambda functions are in order
    for dm in dim_groups.keys():
        for key, var in dim_groups[dm].items():
            # if the dist is uniform for discrete vals like c's and K's
            if var._dist_type == "uniform_int":
                _low = var._dist_params.get("low")  # Default if missing
                _high = var._dist_params.get("high")  # Default if missing

                # Use default parameter capture to correctly bind values
                _foo = lambda l=_low, h=_high: np.random.randint(l, h)
                var._dist_func = _foo

            # if the dist is shifted uniform for discrete vals like c's + K's
            if var._dist_type == "shifted_uniform_int":
                _low = var._dist_params.get("low")
                _high = var._dist_params.get("high")

                # b is fixed shift value
                # Use default parameter capture to correctly bind values
                _foo = lambda b, l=_low, h=_high: b + np.random.randint(l, h)
                var._dist_func = _foo

            # if the dist is uniform for continuous vals like miu's and lambda's
            if var._dist_type == "uniform":
                _low = var._dist_params.get("low")
                _high = var._dist_params.get("high")

                _foo = lambda l=_low, h=_high: np.random.uniform(l, h)
                var._dist_func = _foo

            # with exponential distribution
            if var._dist_type == "exponential":
                _scale = var._dist_params.get("scale")
                var._dist_func = lambda s=_scale:  np.random.exponential(s)

            # if the dist is constant, like the error in the queue model
            if var._dist_type == "constant":
                _constant = var._dist_params.get("constant")
                var._dist_func = lambda cst=_constant: cst

            # if the dist is adjusted uniform for continous vals like chi's = lambda's * (1 - x%)
            if var._dist_type == "adjusted_rate":
                var._dist_func = lambda rate, err: adjust_rate(rate, err)

            # if the dist is a queue theory formula
            if var._dist_type == "queue_length":
                _foo = lambda la_z, mu, c, K: queue_length(la_z, mu, c, K)
                var._dist_func = _foo

            if var._dist_type == "queue_time":
                var._dist_func = lambda La_z, lq: queue_time(La_z, lq)

            if verbose:
                _msg = f"\tConfigured Variable: {key}: {var}"
                print(_msg)
    return dim_groups

In [ ]:
def create_dimensional_matrix(config: pd.DataFrame,
                              dim_groups: dict,
                              framework: str,
                              dim_scheme: DimScheme,
                              verbose: bool = False) -> dict:
    dimensional_model_groups = {}
    for group in config["dm"].unique().tolist():
        group_vars = dim_groups[group]
        if verbose:
            _msg = f"Dimensional Matrix: {group}, "
            _msg += f"with {len(group_vars)} variables."
            print(_msg)
        _descript = f"TAS Dimensional Matrix {group}"
        dimensional_model_groups[group] = DimMatrix(_fwk=framework,
                                                    _idx=group,
                                                    _framework=dim_scheme,
                                                    description=_descript)
        # setting Dimensional Model variables
        dimensional_model_groups[group].variables = group_vars
        # setting Dimensional Model relevance list
        dimensional_model_groups[group].relevant_lt = group_vars
        if verbose:
            n_vars = len(dimensional_model_groups[group].variables)
            n_relev = len(dimensional_model_groups[group].relevant_lt)
            _msg = f"\tDimensional Matrix {group} with {n_vars} variables, "
            _msg += f"and {n_relev} relevant variables."
            print(_msg)
    return dimensional_model_groups

In [ ]:
def solve_dimensional_model(model_groups: dict,
                            verbose: bool = False,
                            details: bool = False) -> dict:

    # solving each dimensional model
    for group, model in model_groups.items():
        # Here you would implement the logic to solve the dimensional model

        model.create_matrix()
        model.solve_matrix()

        if verbose:
            _msg = f"\tDimensional Matrix/Model ID: {group}, "
            _msg += f"with No. Coefficients: {len(model.coefficients)}"
            print(_msg)
            print("\tCreating Matrix for Dimensional Model...")
            print("\tSolving Matrix for Dimensional Model...")
            print(f"\tFinished Solving Dimensional Model: {group}")

        # display model coefficients details
        if details:
            print(f"\tModel {group}, Coefficients Details:")
            for k, v in model.coefficients.items():
                print(f"\t\t'{k}': {fmt(v.pi_expr)}, [{v.var_dims}]")
            print()
    # returning the solved model groups
    return model_groups

In [ ]:
def create_sensitivity_analysis(model_groups: dict) -> dict:
    """Create sensitivity analysis for each dimensional model group.

    Args:
        model_groups (dict): A dictionary containing the groups of dimensional models.

    Returns:
        dict: A dictionary containing the sensitivity analysis results for each model group.
    """

    sensitivity_groups = {}
    # creating sensitivity groups
    for group, models in model_groups.items():
        # add sensitivity analysis handler for each model group
        sensitivity_groups[group] = SensitivityHandler(
            _idx=group,
            _sym=f"$SA_{{TAS_{group}}}$",
            name=f"Sensitivity Analysis for TAS DM {group}",
            description=f"Sensitivity Analysis for Dimensional Model: {group}.",
            _variables=model_groups[group].variables,
            _coefficients=model_groups[group].coefficients,
        )
    # Return the sensitivity analysis results for each model group
    return sensitivity_groups

In [ ]:
def execute_sensitivity_analysis(sens_groups: dict,
                                 vtype: str = None,
                                 n_samples: int = 0,
                                 verbose: bool = False,
                                 details: bool = False) -> dict:

    # executing sensitivity analysis for each sensitivity group
    for group, sens in sens_groups.items():
        if verbose:
            _msg = f"\tExecuting Sensitivity Analysis for Group: {group}."
            _msg += f"with {len(sens.variables)} variables."
            print(_msg)

        # perform symbolic sensitivity analysis
        if vtype is not None:
            if details:
                _msg = f"\tExecuting symbolic analysis for '{vtype}' "
                _msg += f"variables in group: {group}."
                print(_msg)
            sens.analyze_symbolic(val_type=vtype)

        # perform numeric sensitivity analysis
        if n_samples > 0:
            if details:
                _msg = f"\tExecuting numeric analysis with {n_samples} samples "
                _msg += f"in group: {group}."
                print(_msg)
            sens.analyze_numeric(n_samples=n_samples)
        if verbose:
            print(f"\tFinished Sensitivity Analysis for Group: {group}.")
    return sens_groups

In [ ]:
def create_monte_carlo_simulation(model_groups: dict,
                                  n_experiments: int = 1000,
                                  verbose: bool = False) -> dict:
    """Create Monte Carlo simulation for each dimensional model group.

    Args:
        model_groups (dict): A dictionary containing the groups of dimensional models.
        n_experiments (int, optional): The number of experiments to run. Defaults to 1000.
        verbose (bool, optional): Whether to print detailed information. Defaults to False.

    Returns:
        dict: A dictionary containing the Monte Carlo simulation results for each model group.
    """

    simulation_groups = {}
    # creating simulation groups
    for group, models in model_groups.items():
        # add monte carlo simulation handler for each model group
        simulation_groups[group] = MonteCarloHandler(
            _idx=group,
            _sym=f"$MCS_{{TAS_{group}}}$",
            name=f"Monte Carlo Simulation for TAS DM {group}",
            description=f"Monte Carlo Simulation for Dimensional Model: {group}.",
            _variables=models.variables,
            _coefficients=models.coefficients,
            _iterations=n_experiments,
        )
        if verbose:
            _msg = f"\tCreated Monte Carlo Simulation for Group: {group}."
            _msg += f"with {len(models.variables)} variables."
            print(_msg)

    # Return the monte carlo simulation results for each model group
    return simulation_groups

In [ ]:
def execute_monte_carlo(mc_groups: dict,
                        n_samples: int = 1000,
                        verbose: bool = False) -> dict:
    """Execute Monte Carlo simulation for each dimensional model group.

    Args:
        mc_groups (dict): A dictionary containing the groups of Monte Carlo simulations.
        n_experiments (int, optional): The number of experiments to run. Defaults to 1000.
        verbose (bool, optional): Whether to print detailed information. Defaults to False.

    Returns:
        dict: A dictionary containing the results of the Monte Carlo simulations for each model group.
    """

    # executing monte carlo simulation for each simulation group
    for group, mc_handler in mc_groups.items():
        if verbose:
            _msg = f"\tExecuting Monte Carlo Simulation for Group: {group} "
            _msg += f"with {len(mc_handler.variables)} variables."
            print(_msg)

        # configure distributions and simulations
        mc_handler._config_distributions()
        mc_handler._config_simulations()
        # perform monte carlo simulation
        mc_handler.simulate(n_samples=n_samples)

        mc_groups[group] = mc_handler
        if verbose:
            print(f"\tFinished Monte Carlo Simulation for Group: {group}.")
    return mc_groups

In [ ]:
def process_monte_carlo_results(simulation_groups: dict,
                                verbose: bool = False) -> tuple:
    """Post-process Monte Carlo simulation results for each dimensional model group.

    Args:
        simulation_groups (dict): A dictionary containing the groups of Monte Carlo simulations.

    Returns:
        dict: A dictionary containing the post-processed results for each group.
    """
    # coefficient global index
    i = 0

    # records for the unifided simulation dataframe
    records = {}

    # simulation results statistics
    statistics = []

    # global coefficient name = coefficient formula
    coefs = {}

    for group, sim in simulation_groups.items():
        if verbose:
            _msg = f"\tPost-Processing Monte Carlo Results for Group: {group}, "
            _msg += f"with {len(sim.variables)} variables, "
            _msg += f"and {sim._iterations} iterations."
            print(_msg)
            _msg = f"\tVariables: {list(sim.variables.keys())}, "
            _msg += f"No. Coefficients: {len(sim.coefficients)}"
        for j, (pi, coef) in enumerate(sim.coefficients.items()):
            # getting the coefficient data
            if verbose:
                _msg = f"\t\tProcessing Coefficient: {coef.name}: {pi} "
                _msg += f"= {coef.pi_expr}"
                print(_msg)
                _msg = f"\t\tGlobal Idx: {i}, Local Idx: {j}, "
                _msg += f"Inputs := {list(coef.var_dims.keys())}, "
                _msg += f"Size: {len(coef.var_dims)}"
                print(_msg)

            # rename Pi for DataFrame column with global idx
            pi_rename = str(pi)
            pi_rename = pi_rename.replace(str(j), str(i))
            if verbose:
                _msg = f"\t\tRenaming Coef. FROM: {pi} TO: {pi_rename}"
                print(_msg)
            
            coefs[pi_rename] = coef.pi_expr

            # get simulation results
            simulations = sim.get_simulation(pi)
            # print(type(simulations))
            if simulations:

                # exporting results
                results = simulations.extract_results()

                # rename keys in results dict
                for key in list(results.keys()):
                    new_key = key.replace(pi, pi_rename)
                    results[new_key] = results.pop(key)
                
                records.update(results)

                # exporting statistics
                stats = {
                    "name": pi_rename,
                    "coef": coef.pi_expr
                }
                # update with simulation statistics
                stats.update(simulations.statistics)
                # add to statistics list
                statistics.append(stats)
            # updating global idx
            i += 1
        if verbose:
            _msg = f"\tFinished Post-Processing Coefficients Group: {group}.\n"
            print(_msg)

    # creating DataFrame for records and statistics
    records = pd.DataFrame(records)
    statistics = pd.DataFrame(statistics)

    return records, statistics, coefs

In [ ]:
def consolidate_simulation_results(coefficient_groups: pd.DataFrame,
                                   columns: list,
                                   verbose: bool = False,
                                   details: bool = False) -> pd.DataFrame:
    """Consolidate simulation results from multiple coefficient groups.

    Args:
        coefficient_groups (pd.DataFrame): A DataFrame containing the simulation results for different coefficient groups.
        columns (list): A list of column names to process.
        verbose (bool, optional): If True, print detailed processing information. Defaults to False.
        details (bool, optional): If True, print additional details about the processing. Defaults to False.

    Returns:
        pd.DataFrame: A DataFrame containing the consolidated simulation results.
    """
    records = {}
    for i, col in enumerate(columns):
        if verbose:
            print(f"\tProcessing Column {i+1}/{len(columns)}: {col}")
        # iterate over the pi dataframe columns
        for j in range(i, coefficient_groups.shape[1], len(columns)):
            # check if the formula matches
            key = coefficient_groups.columns[j]
            val = coefficient_groups[key].values
            # if the column name is new, add it
            if col not in records:
                records[col] = val
                if details:
                    print(f"\t\tAdding New Column: {key}")
            # if the column name already exists, append data
            elif col in records:
                records[col] = np.concatenate(([records[col], val]))
        if verbose:
            print(f"\tFinished Processing Column: {col}\n")

    if details:
        print("Consolidated Records Summary:")
        for k, v in records.items():
            print(f"\t{k}: {len(v)}")

    return pd.DataFrame(records)

In [ ]:
# path = os.path.dirname(__file__)\
PATH = os.getcwd()
print(f"Notebook path: {PATH}")

In [ ]:
# Folder names
asset_folder = "assets"
docs_folder = "docs"
img_folder = "img"
data_folder = "data"
report_folder = "reports"
results_folder = "results"
cs_folder = "cs1"

# defining the contour lines for plots
contour_vals = [
    0.05,   # 0
    0.1,    # 1
    0.2,    # 2
    0.3,    # 3
    0.4,    # 4
    0.5,    # 5
    0.6,    # 6
    # 0.65,   # 7
    0.7,    # 8
    0.8,    # 9
    0.9,    # 10
    0.95,   # 11
    0.99,   # 12
    # 1.0,    # 13
    # 1.2,    # 14
    # 1.5,    # 15
    # 2.0,    # 16
    # 3.0,    # 17
]

In [ ]:
# setting case study data folder
file_path = os.path.join(PATH, data_folder, cs_folder)
print(f"Data path: {file_path}")

### **Queue Model**
#### **Creating Dimensionless Charts**

##### **FDUs (Fundamental Design Units)**

In [ ]:
# setting up custom FDU for CUSTOM architecture
tas_fwk = "SOFTWARE"
tas_scm = DimScheme(_sym="FDU_{TAS}",
                    _alias="fdu_tas",
                    _idx=0,
                    name="TAS FDUs",
                    _fwk=tas_fwk,
                    description="FDU schema for the TAS case study.")
# print(tas_scm)
print("=== TAS Fundamental Dimenional Unit (FDU) Schema ===")
for dim in tas_scm.fdus:
    print(f"\t'{dim.sym}': {dim.name}, [{dim.unit}]")

In [ ]:
tas_fwk = "CUSTOM"
fdu_lt = [
    {
        "_idx": 0,
        "_sym": "T",
        "_fwk": tas_fwk,
        "description": "Duration of an event or interval.",
        "_unit": "s",
        "name": "Time",
        },
    {
        "_idx": 1,
        "_sym": "I",
        "_fwk": tas_fwk,
        "description": "Computational unit to process/handle or store requests.",
        "_unit": "req",
        "name": "Item",
        },
    {
        "_idx": 2,
        "_sym": "D",
        "_fwk": tas_fwk,
        "description": "Information processed by a component.",
        "_unit": "bit",
        "name": "Data",
    },
]
tas_scm = DimScheme(_sym="FDU_{Custom TAS}",
                    _alias="fdu_custom_tas",
                    _idx=0,
                    name="Custom TAS FDUs",
                    _fdus=fdu_lt,
                    _fwk=tas_fwk,
                    description="Minimal FDU schema for TAS case study.")

# print(tas_scm)
print("=== TAS Fundamental Dimensional Unit (FDU) Schema ===")
for dim in tas_scm.fdus:
    print(f"\t'{dim.sym}': {dim.name}, [{dim.unit}]")

In [ ]:
tas_scm.update_global_config()
print("---- Checking Schema Regex ----")
print("\tWKNG_DFLT_FDU_PREC_LT:", config.WKNG_FDU_PREC_LT)
print("\tWKNG_FDU_RE:", config.WKNG_FDU_RE)
print("\tWKNG_POW_RE:", config.WKNG_POW_RE)
print("\tWKNG_NO_POW_RE:", config.WKNG_NO_POW_RE)
print("\tWKNG_FDU_SYM_RE:", config.WKNG_FDU_SYM_RE)

In [ ]:
print("---- Config Sensitivty & Monte Carlo Samples ----")
n_sens = 1000
n_exp = 1000
print(f"\tSensitivity Analysis Samples: {n_sens}")
print(f"\tMonte Carlo Simulation Samples: {n_exp}")

##### **Base Configuration**

In [ ]:
# Load configuration with mixed queue models
dflt_qn_cfg = load(file_path, "default_qn_model.csv")
print("Queue Network Configuration:")
dflt_qn_cfg.head()

In [ ]:
# Load configuration with mixed queue models
dflt_da_cfg = load(file_path, "default_dim_variables.csv")
print("Dimension Variables Configuration:")
dflt_da_cfg.head()

###### **Loading Dimensional Variables**

In [ ]:
print("--- Dimensional Var Groups by Dimensional Matrix ---")

# create a dimensional set of variables from config file
dflt_dim_var_groups = create_dimensional_groups(dflt_da_cfg,
                                                verbose=False)

print(f"--- No. of Dimensional Var Groups: {len(dflt_dim_var_groups)} ---")

In [ ]:
print("--- Configure Simulation Distribution Function for Variables ---")
# configuring distribution functions for dimensional variables

dflt_dim_var_groups = configure_dimensional_dist(dflt_dim_var_groups,
                                                 verbose=False)

print(f"--- No. of Dimensional Var Groups: {len(dflt_dim_var_groups)} ---")

###### **Creating Dimensional Model**

In [ ]:
print("--- Creating Dimensional Model (Matrix) ---")
print(f"framework: {tas_fwk}, scheme: {tas_scm.sym}\n")
dflt_dim_model_groups = create_dimensional_matrix(dflt_da_cfg,
                                                  dflt_dim_var_groups,
                                                  tas_fwk,
                                                  tas_scm,
                                                  verbose=True)

n = len(dflt_dim_model_groups)
print(f"\n--- No. of Dimensional Model Groups: {n} ---")

In [ ]:
print("--- Solving Dimensional Model (Matrix) ----")

dflt_dim_model_groups = solve_dimensional_model(dflt_dim_model_groups,
                                               verbose=True,
                                               details=True)

n = len(dflt_dim_model_groups)
print(f"\n--- No. of Dimensional Model Groups: {n} ---")

###### **Calculating Pi-Coefficients**

In [ ]:
print("--- Creating the Sensitivity Groups ---")
print("--- Indexing Sensitivity Groups ---")

# Sensitivity Groups
dflt_sens_groups = {}

# create sensitivity analysis for each dimensional model group
dflt_sens_groups = create_sensitivity_analysis(dflt_dim_model_groups)

print(f"--- No. of Sensitivity Groups: {len(dflt_sens_groups)} ---")

###### **Running Sensitivity Analysis**

In [ ]:
print("--- Executing Sensitivity Analysis ---")

# executing sensitivity analysis for each sensitivity group
dflt_sens_groups = execute_sensitivity_analysis(dflt_sens_groups,
                                                vtype="mean",
                                                n_samples=n_sens,
                                                verbose=True,
                                                details=False,)

print(f"--- No. of Sensitivity Groups: {len(dflt_sens_groups)} ---")

In [ ]:
print("---- Sensitivity Analysis Post-Processing ----")
# detailed report
# coefficient global index
i = 0

# sensitivity report statistical data
sens_records = []

# global coefficient name = coefficient formula
pi_coef = {}

for dm in dflt_sens_groups:
    print(f"Sensitivity Report for Dimensional Model ID: {dm}")
    n = len(dflt_sens_groups[dm].results)
    print(f"\tSensitivity Reports Size: {n}")
    for key, val in dflt_sens_groups[dm].results.items():
        print(f"\t{key}: {val}")
    print(f"\tEnding Sensitivity report No. {dm}\n")
    # TODO complete this part later, correctly indexing all coefficients

# creating DataFrame from records
dflt_node_sens = pd.DataFrame(sens_records)


###### **Running Monte Carlo Simulation**

In [ ]:
print("--- Create Monte Carlo Simulations ---")
dflt_mc_groups = {}

dflt_mc_groups = create_monte_carlo_simulation(dflt_dim_model_groups,
                                               n_experiments=n_exp,
                                               verbose=True)

print(f"--- No. of Monte Carlo Simulation Groups: {len(dflt_mc_groups)} ---")

In [ ]:
print("--- Executing Monte Carlo Simulations ---")

dflt_mc_groups = execute_monte_carlo(dflt_mc_groups,
                                     n_samples=n_exp,
                                     verbose=True)

print(f"--- No. of Monte Carlo Simulation Groups: {len(dflt_mc_groups)} ---")

###### **Plotting Dimensional Model Chart**

In [ ]:
print("--- Monte Carlo Simulation Post-Processing (Exp + Stats) ----")

# Call the function to process results
answer = process_monte_carlo_results(dflt_mc_groups,
                                     verbose=True)
dflt_mc_exp, dflt_mc_stats, pi_coef = answer

print(f"--- No. of Monte Carlo Simulation Records: {len(dflt_mc_exp)} ---")
print(f"--- No. of Monte Carlo Simulation Statistics: {len(dflt_mc_stats)} ---")
print(f"--- No. of Monte Carlo Simulation Coefficients: {len(pi_coef)} ---")


In [ ]:
print("--- Saving Monte Carlo Simulation Experiment Results ---")
# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_node_exp.csv", dflt_mc_exp)

print(dflt_mc_exp.shape)
dflt_mc_exp.info()
dflt_mc_exp.head()

In [ ]:
print("--- Saving Monte Carlo Simulation Statistics ---")
# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_node_stats.csv", dflt_mc_stats)

print(dflt_mc_stats.shape)
dflt_mc_stats.info()
dflt_mc_stats.head()

In [ ]:

print("--- Renaming Pi-Coefficients from DataFrame ---")
# Extracting just the coefficients from the experimental DataFrame
pi_keys = list(pi_coef.keys())
print(f"Total No. of Pi-Coefficients: {len(pi_keys)}")
dflt_pi_coefs = pd.DataFrame(dflt_mc_exp[pi_keys])

pi_cols = {}
# renaming columns with coef = formula
for k, v in pi_coef.items():
    pi_cols[k] = f"{k}={v}"

# renaming columns
dflt_pi_coefs.rename(columns=pi_cols, inplace=True)
print("--- Finished Renaming Pi-Coefficients from DataFrame ---")

In [ ]:
print("--- Saving Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_node_coeffs.csv", dflt_pi_coefs)

# checking the Pi DataFrame
print(dflt_pi_coefs.shape)
dflt_pi_coefs.info()
dflt_pi_coefs.head()

In [ ]:
# make generic columns for dimensionless coefficients
print("--- Creating Generic Columns for Dimensionless Coefficients ---")
sys_name = "TAS"
sys_cols = [re.sub(r"\d+", sys_name, col) for col in pi_coef.values()]

# removing redundant entries, keeping order
sys_cols = list(dict.fromkeys(sys_cols))
for i in range(len(sys_cols)):
    formula = sys_cols[i]
    sys_cols[i] = f"\\Pi_{{{i+1}}}={formula}"

print(f"System generic columns: {len(sys_cols)}, {sys_cols}")

In [ ]:
print("--- Post-Processing System Pi-Coefficients ---")
print(f"Pi-Coefficients shape: {dflt_pi_coefs.shape}")
# dictionary for system records
dflt_sys_coef = consolidate_simulation_results(dflt_pi_coefs,
                                               sys_cols,
                                               verbose=False,
                                               details=False)

print(f"System Pi-Coefficients shape: {dflt_sys_coef.shape}")


In [ ]:
print("--- Checking System Records Info ---")
print("--- System DataFrame ---")
print(dflt_sys_coef.shape)
dflt_sys_coef.info()
dflt_sys_coef.head()

In [ ]:
# creating derived Pi-Coefficients
print("---- Creating Derived Pi-Coefficients ----")
# derived coefficients columns
# defining plot colums
plot_cols = []

# >>> working variables for derived a new coefficient
# Service Effectiveness Coef: Pi_6 = Pi_2 * Pi_3^{-1} = chi / mu
name = "Eff"
print(f"Creating System Congestion Coefficient: \\Pi_{{{name}}}")

# Pi_2 = mu / lambda
temp = f"\\Pi_{{2}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_2 = dflt_sys_coef[temp]

# Pi_3 = chi / lambda
temp = f"\\Pi_{{3}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_3 = dflt_sys_coef[temp]

numerator = "\\chi"
denominator = "\\mu"

# Pi_8 = chi * lambda / (mu * lambda) = chi / mu
eff_coef = f"\\eta="
eff_coef += f"\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
dflt_sys_coef[eff_coef] = pi_2 * (pi_3**-1)

# >>> working variables for derived a new coefficient
# normalizing P_1 by another coefficient to improve readbility
# Stall Coefficient: Pi_7 = P_1 * P_4^(-1) = lambda * W / K
# Adjusted in log-space: Ln(P_7 + 1 )= Ln(lambda * W / K + 1)
name = "Stall"
print(f"Creating System Stall Coefficient: \\Pi_{{{name}}}")

# Pi_1 = lambda * W / c
numerator = f"\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
denominator = f"c_{{{sys_name}}}"
temp = f"\\Pi_{{1}}=\\frac{{{numerator}}}{{{denominator}}}"
pi_1 = dflt_sys_coef[temp]

# Pi_4 = K / c
temp = f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_4 = dflt_sys_coef[temp]

# Pi_7 = Lambda * W / c -> Pi_7 = ln(Pi_7 + 1)
stall_coef = f"S=\\Pi_{{{name}}}=\\ln(\\frac{{\\lambda*W}}{{K}} + 1)"
pi_7 = pi_1 * (pi_4**-1)
dflt_sys_coef[stall_coef] = np.log(pi_7 + 1)

# >>> working variables for derived a new coefficient
# Occupation Coefficient: Pi_8 = Pi_5 * Pi_4^{-1} = L / K
name = "Occ"
print(f"Creating System Occupancy Coefficient: \\Pi_{{{name}}}")

# Pi_4 = L / c
temp = f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_4 = dflt_sys_coef[temp]

# Pi_5 = L / c
temp = f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_5 = dflt_sys_coef[temp]

numerator = "L"
denominator = "K"

# Pi_8 = L * c / (K * c) = L / K
occ_coef = f"O=\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
dflt_sys_coef[occ_coef] = pi_5 * (pi_4**-1)

# change order to change chart distribution
# x-axis, y-axis, z-axis/contourn
plot_cols.append(occ_coef)
plot_cols.append(stall_coef)
plot_cols.append(eff_coef)

# checking derived coefficients
print(f"Derived Coefficients: {len(plot_cols)}, {plot_cols}")

# sort by first colum from lower to highest value
# dflt_sys_coef.sort_values(by=plot_cols[0], inplace=True)
# dflt_sys_coef.reset_index(drop=True, inplace=True)

In [ ]:
print("\n--- System Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "dflt_dimensional_net_coeffs.csv", dflt_pi_coefs)
dflt_sys_coef.info()
dflt_sys_coef.head()

In [ ]:
# create plot dataframe
plot_df = pd.DataFrame(dflt_sys_coef[plot_cols])

In [ ]:
print("--- Charting Occupation and Congestion Coefficients ---")

# using contour lines at the beginning of the program
metrics = plot_df.columns.tolist()
labels = [
    "Occupation",       # "Effectiveness",
    "Stall",            # "Stall",
    "Efficiency",       # "Occupation",
]
contour_name = plot_cols[-1]
print(f"Contour levels from: {contour_name}")
print(f"No. of contour lines {len(contour_vals)}")
plot_df.info()

In [ ]:
print("--- Cleaning Invalid Values from Plot Data ---")
# clear plot data
# clear x-axis: occ <= 1
LIM_FLOW = 1.0
LIM_BUFFER = 1.0
print(f"cleaning invalid values in: {eff_coef}")
plot_df = plot_df[plot_df[eff_coef] <= LIM_FLOW]
print(f"Plot data with {eff_coef} < {LIM_FLOW}: {plot_df.shape}")

# # clear y-axis: stall <= K/c <= something when K is max and c is min
# print(f"cleaning invalid values in: {stall_coef}")
# plot_df = plot_df[plot_df[stall_coef] <= (K_MAX / C_MIN)]
# print(f"Plot data with {stall_coef} < {K_MAX}/{C_MIN}: {plot_df.shape}")

# clear contour/z-axis: eta <= 1
print(f"cleaning invalid values in: {occ_coef}")
plot_df = plot_df[plot_df[occ_coef] <= LIM_BUFFER]
print(f"Plot data with {occ_coef} < {LIM_BUFFER}: {plot_df.shape}")
plot_df.info()

In [ ]:
# sorting by Occ, them by Stall, then by Eff
print(f"Sorting plot data by: {plot_cols}")
plot_df.sort_values(by=plot_cols, inplace=True)
plot_df.reset_index(drop=True, inplace=True)
plot_df.info()

In [ ]:
# plotting the queue network dimensionless chart
# selecting images folder
print()
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# plot dimensionless system chart
title = "TAS Performance Chart: Initial Configuration"
plot_performance_coef_chart(plot_df,
                            contour_name,
                            contour_vals,
                            metrics,
                            labels,
                            title,
                            file_path,
                            "dflt_dimensional_perf_chart.png",
                            percentile={"x": [0.01, 0.99],
                                        "y": [0.01, 0.99]},
                            scale={"x": "log", "y": "log"},
                            limits={"x": [1e-5, 1.0],
                                    "y": [1.9e-2, 1.3e2]},)

In [ ]:

print(metrics)
print(labels)

# plot dimensionless experiments
title = "TAS Performance Chart: 3D analysis, Full Experiments"
plot_experiment_coef_chart(plot_df,
                           metrics,
                           labels,
                           title,
                           file_path,
                           "dflt_dimensional_exp_chart.png",
                           percentile={"x": [0.01, 0.99],
                                       "y": [0.01, 0.99],
                                       "z": [0.01, 0.99],},
                           scale={"x": "log", "y": "log", "z": "log"},
                           limits={"x": [1e-5, 1.0],
                                   "y": [1.9e-2, 1.0e1],
                                   "z": [1e-5, 1.0]},)

In [ ]:
# plot this data in regular chart
plot_df.info()
print(plot_df.describe())

###### **Using the Dimensionless Chart**

In [ ]:


# defining training df to use later
dflt_train_df = pd.DataFrame(plot_df)

In [ ]:
a = 

In [ ]:
a =

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

# === Example: dimensionless coefficients from your chart ===
# X = χ / μ  (occupation ratio)
# Y = λ * W / c  (normalized waiting time)
# Z = L / K  (efficiency contour)

# # Simulated Monte Carlo samples (replace with your real data)
# np.random.seed(42)
# X = np.random.uniform(0.1, 1.0, 5000)
# Y = np.random.uniform(0.1, 2.0, 5000)
# Z = (X * Y) / (1 + X**2)  # Example nonlinear relation (you’ll use your L/K data)


data = plot_df[[occ_coef, eff_coef, stall_coef]]
data = data.sample(frac=0.99, replace=False, random_state=42)
data = data.to_numpy()
X, Y, Z = data[:, 0], data[:, 1], data[:, 2]



# === Step 1. Fit a polynomial regression surface ===
degree = 3  # start small to avoid overfitting
model_sk = make_pipeline(PolynomialFeatures(degree), LinearRegression())
XY = np.column_stack((X, Y))
model_sk.fit(XY, Z)

# === Step 2. Predict/interpolate new configurations ===
X_pred, Y_pred = np.meshgrid(
    np.linspace(0, 1, 1000),
    np.linspace(0, 1, 1000)
)
Z_pred = model_sk.predict(np.column_stack(
    (X_pred.ravel(), Y_pred.ravel()))).reshape(X_pred.shape)

# === Step 3. Visualize the interpolated surface ===
plt.figure(figsize=(7, 5))
cp = plt.contourf(X_pred, Y_pred, Z_pred, levels=50, cmap="viridis")
plt.colorbar(cp, label=r"Stall: $ln(\lambda W / c + 1)$")
plt.xlabel(r"Occupation: L/K")
plt.ylabel(r"Efficiency: $\chi / \mu$")
plt.title("Interpolated Dimensionless Efficiency Surface")
plt.show()

In [ ]:
# === Step 4. Predict new unseen configurations ===
new_config = np.array([[0.8204, 0.2290]])  # (χ/μ, λW/K)
predicted_stall = model_sk.predict(new_config)
print(f"Predicted L/K for {new_config}: {predicted_stall[0]:.4f}")
c = 1
la_z = 38.72
W = (c * (np.exp(predicted_stall) - 1)) / la_z
print(f"Predicted W for λ={la_z}, c={c}, Wait Time: {W[0]:.4f} s")

In [ ]:
import numpy as np
from scipy.interpolate import Rbf
import matplotlib.pyplot as plt

# Example: Load Monte Carlo results
# X = efficiency (chi/mu)
# Y = utilization (lambda * W / c)
# Z = occupancy ratio (L/K)
# data = np.loadtxt("dimensionless_results.csv", delimiter=",")
data = plot_df[[occ_coef, eff_coef, stall_coef]]
data = data.sample(frac=0.5, replace=False, random_state=42)
data = data.to_numpy()

X, Y, Z = data[:, 0], data[:, 1], data[:, 2]

# # Build interpolation model (Radial Basis Function)
# model_Rbf = Rbf(X, Y, Z,
#                 function="multiquadric",
#                 smooth=0.1)

# When creating your RBF model, try different kernels and parameters:
model_Rbf = Rbf(X, Y, Z,
                function='inverse',  # Try 'multiquadric', 'inverse', 'gaussian'
                epsilon=0.2,         # Adjust this parameter for smoothness
                smooth=0.1)          # Add some smoothing for noisy data

# model_Rbf = Rbf(X, Y, Z,
#                 function="gaussian",
#                 epsilon=0.1,
#                 smooth=0.01)

# Now we can interpolate any (X, Y)
Xq, Yq = np.meshgrid(np.linspace(min(X), max(X), 250),
                     np.linspace(min(Y), max(Y), 250))
Zq = model_Rbf(Xq, Yq)

# Visualize the interpolated surface
plt.figure(figsize=(8, 6))
contour = plt.contourf(Xq, Yq, Zq, levels=30, cmap="viridis")
plt.colorbar(contour, label="Stall: ln((λ*W/K) + 1)")
plt.xlabel("Occupancy: (L/K)")
plt.ylabel("Efficiency: (χ/μ)")
plt.title("Interpolated Dimensionless Surface")
plt.show()

In [ ]:
def solve_dimensionless_time_log(dim_model, la_z, mu, K, L, error=0.0, verbose=False):
    """Improved version working in log space for better interpolation"""
    _chi = la_z * (1 + error)

    if verbose:
        print(f"departure rate aprox (X): {_chi}")

    # Calculate dimensionless parameters
    pi_1 = L / K  # Occupation
    pi_2 = _chi / mu  # Efficiency

    if verbose:
        print(f"pi_1 (Occ): {pi_1}, pi_2 (Eff): {pi_2}")

    # Transform inputs to log space for better interpolation
    log_pi_1 = np.log10(max(pi_1, 1e-10))  # Avoid log(0)
    log_pi_2 = np.log10(max(pi_2, 1e-10))

    try:
        # Predict in log space
        log_pi_3 = float(dim_model(log_pi_1, log_pi_2))
        # Convert back from log space
        pi_3 = 10**log_pi_3

        # Convert from ln(λW/K + 1) back to W
        W = (K / la_z) * (np.exp(pi_3) - 1)

        if verbose:
            print(f"Dimensionless Coefficient pi_3 (Stall): {pi_3}")
    except Exception as e:
        print(f"Error solving for W: {e}")
        W = -1.0

    if verbose:
        print(f"Dimensionless time (W): {W}")

    return W

In [ ]:
def solve_dimensionless_time(dim_model: Rbf,
                             la_z: float,
                             mu: float,
                             K: int,
                             L: float,
                             #  c: int = 1,
                             error: float = 0.0,
                             verbose: bool = False) -> float:
    """solve_dimensionless_time _summary_

    Args:
        dim_model (Rbf): _description_
        la_z (float): _description_
        mu (float): _description_
        K (int): _description_
        L (float): _description_
        c (int, optional): _description_. Defaults to 1.
        error (float, optional): _description_. Defaults to 0.0.
        verbose (bool, optional): _description_. Defaults to False.

    Returns:
        float: _description_
    """
    _chi = la_z * (1 + error)
    print(f"departure rate aprox (X): {_chi}")
    # Occ = pi_1: occupation
    pi_1 = L / K
    # Eff = pi_2: effectiveness
    pi_2 = _chi / mu
    print(f"pi_1 (Occ): {pi_1}, pi_2 (Eff): {pi_2}")
    # Stall = pi_3: stall
    # Calculate dimensionless time W
    W, pi_3 = -1.0, -1.0

    # solving for W using the dimensional model
    try:
        pi_3 = float(dim_model(pi_1, pi_2))
        # Correct conversion: Convert from ln(λW/K + 1) back to W
        W = (K / la_z) * (np.exp(pi_3) - 1)
        if verbose:
            _msg = f"Dimensionless Coefficient pi_3 (Stall): {pi_3}"
            print(_msg)
    except Exception as e:
        print(f"Error solving for W: {e}")
        W = -1.0
    
    if verbose:
        print(f"Dimensionless time (W): {W}")
    return W

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Create and train model
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_model.fit(np.column_stack((X, Y)), Z)

# Create prediction function with same interface as RBF


def gb_predict(occ, eff):
    return gb_model.predict([[occ, eff]])[0]


# Then use this prediction function in your solver
result = solve_dimensionless_time(
    gb_predict, la_z=la_z, mu=mu, K=K, L=L, error=error)

In [ ]:
# Create a grid for visualization
xx = np.linspace(min(X), max(X), 100)
yy = np.linspace(min(Y), max(Y), 100)
XX, YY = np.meshgrid(xx, yy)
ZZ = np.zeros_like(XX)

# Generate predictions across the grid
for i in range(XX.shape[0]):
    for j in range(XX.shape[1]):
        ZZ[i, j] = model_Rbf(XX[i, j], YY[i, j])  # Replace with your model

# Plot original data points and interpolated surface
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111)

# Plot interpolated surface
contour = ax.contourf(XX, YY, ZZ, levels=30, cmap='viridis', alpha=0.8)
plt.colorbar(contour, label="Stall coefficient")

# Plot original data points
scatter = ax.scatter(X, Y, c=Z, s=20, cmap='viridis', edgecolor='k', alpha=0.6)

# Add log scales if working in log space
ax.set_xscale('log')
ax.set_yscale('log')

plt.title('Interpolation Quality Check')
plt.xlabel('Occupation (L/K)')
plt.ylabel('Efficiency (χ/μ)')
plt.tight_layout()
plt.show()

In [ ]:
w_test = solve_dimensionless_time(model_Rbf,
                                  la_z=104.30,   # la_z: arrival rate
                                  mu=150.0,     # mu: service rate
                                  K=10,         # K: capacity
                                  L=2.29,       # L: # requests, MEASURED!
                                  error=0.18,   # error %
                                  verbose=True)

##### **Optimized Configuration**

In [ ]:
# setting case study data folder
file_path = os.path.join(PATH, data_folder, cs_folder)
print(f"Data path: {file_path}")

In [ ]:
# Load configuration with mixed queue models
opti_qn_cfg = load(file_path, "optimal_qn_model.csv")
print("Queue Network Configuration:")
opti_qn_cfg.head()

In [ ]:
# Load configuration with mixed queue models
opti_da_cfg = load(file_path, "optimal_dim_variables.csv")
print("Dimension Variables Configuration:")
opti_da_cfg.head()

###### **Loading Dimensional Variables**

In [ ]:
print("--- Dimensional Var Groups by Dimensional Matrix ---")

# create a dimensional set of variables from config file
opti_dim_var_groups = create_dimensional_groups(opti_da_cfg,
                                                verbose=False)

print(f"--- No. of Dimensional Var Groups: {len(opti_dim_var_groups)} ---")

In [ ]:
print("--- Configure Simulation Distribution Function for Variables ---")
# configuring distribution functions for dimensional variables

opti_dim_var_groups = configure_dimensional_dist(opti_dim_var_groups,
                                                 verbose=False)

print(f"--- No. of Dimensional Var Groups: {len(opti_dim_var_groups)} ---")

###### **Creating Dimensional Model**

In [ ]:
print("--- Creating Dimensional Model (Matrix) ---")
print(f"framework: {tas_fwk}, scheme: {tas_scm.sym}\n")
opti_dim_model_groups = create_dimensional_matrix(opti_da_cfg,
                                                  opti_dim_var_groups,
                                                  tas_fwk,
                                                  tas_scm,
                                                  verbose=True)

n = len(opti_dim_model_groups)
print(f"\n--- No. of Dimensional Model Groups: {n} ---")

In [ ]:
print("--- Solving Dimensional Model (Matrix) ----")

opti_dim_model_groups = solve_dimensional_model(opti_dim_model_groups,
                                               verbose=True,
                                               details=True)

n = len(opti_dim_model_groups)
print(f"\n--- No. of Dimensional Model Groups: {n} ---")

###### **Calculating Pi-Coefficients**

In [ ]:
print("--- Creating the Sensitivity Groups ---")
print("--- Indexing Sensitivity Groups ---")

# Sensitivity Groups
opti_sens_groups = {}

# create sensitivity analysis for each dimensional model group
opti_sens_groups = create_sensitivity_analysis(opti_dim_model_groups)

print(f"--- No. of Sensitivity Groups: {len(opti_sens_groups)} ---")

###### **Running Sensitivity Analysis**

In [ ]:
print("--- Executing Sensitivity Analysis ---")

# executing sensitivity analysis for each sensitivity group
opti_sens_groups = execute_sensitivity_analysis(opti_sens_groups,
                                                vtype="mean",
                                                n_samples=n_sens,
                                                verbose=True,
                                                details=False,)

print(f"--- No. of Sensitivity Groups: {len(opti_sens_groups)} ---")

In [ ]:
print("---- Sensitivity Analysis Post-Processing ----")
# detailed report
# coefficient global index
i = 0

# sensitivity report statistical data
sens_records = []

# global coefficient name = coefficient formula
pi_coef = {}

for dm in opti_sens_groups:
    print(f"Sensitivity Report for Dimensional Model ID: {dm}")
    n = len(opti_sens_groups[dm].results)
    print(f"\tSensitivity Reports Size: {n}")
    for key, val in opti_sens_groups[dm].results.items():
        print(f"\t{key}: {val}")
    print(f"\tEnding Sensitivity report No. {dm}\n")
    # TODO complete this part later, correctly indexing all coefficients

# creating DataFrame from records
opti_node_sens = pd.DataFrame(sens_records)


###### **Running Monte Carlo Simulation**

In [ ]:
print("--- Create Monte Carlo Simulations ---")
opti_mc_groups = {}

opti_mc_groups = create_monte_carlo_simulation(opti_dim_model_groups,
                                               n_experiments=n_exp,
                                               verbose=True)

print(f"--- No. of Monte Carlo Simulation Groups: {len(opti_mc_groups)} ---")

In [ ]:
print("--- Executing Monte Carlo Simulations ---")

opti_mc_groups = execute_monte_carlo(opti_mc_groups,
                                     n_samples=n_exp,
                                     verbose=True)

print(f"--- No. of Monte Carlo Simulation Groups: {len(opti_mc_groups)} ---")

###### **Plotting Dimensional Model Chart**

In [ ]:
print("--- Monte Carlo Simulation Post-Processing (Exp + Stats) ----")

# Call the function to process results
answer = process_monte_carlo_results(opti_mc_groups,
                                     verbose=True)
opti_mc_exp, opti_mc_stats, pi_coef = answer

print(f"--- No. of Monte Carlo Simulation Records: {len(opti_mc_exp)} ---")
print(f"--- No. of Monte Carlo Simulation Statistics: {len(opti_mc_stats)} ---")
print(f"--- No. of Monte Carlo Simulation Coefficients: {len(pi_coef)} ---")


In [ ]:
print("--- Saving Monte Carlo Simulation Experiment Results ---")
# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_node_exp.csv", opti_mc_exp)

print(opti_mc_exp.shape)
opti_mc_exp.info()
opti_mc_exp.head()

In [ ]:
print("--- Saving Monte Carlo Simulation Statistics ---")
# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_node_stats.csv", opti_mc_stats)

print(opti_mc_stats.shape)
opti_mc_stats.info()
opti_mc_stats.head()

In [ ]:

print("--- Renaming Pi-Coefficients from DataFrame ---")
# Extracting just the coefficients from the experimental DataFrame
pi_keys = list(pi_coef.keys())
print(f"Total No. of Pi-Coefficients: {len(pi_keys)}")
opti_pi_coefs = pd.DataFrame(opti_mc_exp[pi_keys])

pi_cols = {}
# renaming columns with coef = formula
for k, v in pi_coef.items():
    pi_cols[k] = f"{k}={v}"

# renaming columns
opti_pi_coefs.rename(columns=pi_cols, inplace=True)
print("--- Finished Renaming Pi-Coefficients from DataFrame ---")

In [ ]:
print("--- Saving Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_node_coeffs.csv", opti_pi_coefs)

# checking the Pi DataFrame
print(opti_pi_coefs.shape)
opti_pi_coefs.info()
opti_pi_coefs.head()

In [ ]:
# make generic columns for dimensionless coefficients
print("--- Creating Generic Columns for Dimensionless Coefficients ---")
sys_name = "TAS"
sys_cols = [re.sub(r"\d+", sys_name, col) for col in pi_coef.values()]

# removing redundant entries, keeping order
sys_cols = list(dict.fromkeys(sys_cols))
for i in range(len(sys_cols)):
    formula = sys_cols[i]
    sys_cols[i] = f"\\Pi_{{{i+1}}}={formula}"

print(f"System generic columns: {len(sys_cols)}, {sys_cols}")

In [ ]:
print("--- Post-Processing System Pi-Coefficients ---")
print(f"Pi-Coefficients shape: {opti_pi_coefs.shape}")
# dictionary for system records
opti_sys_coef = consolidate_simulation_results(opti_pi_coefs,
                                               sys_cols,
                                               verbose=False,
                                               details=False)

print(f"System Pi-Coefficients shape: {opti_sys_coef.shape}")


In [ ]:
print("--- Checking System Records Info ---")
print("--- System DataFrame ---")
print(opti_sys_coef.shape)
opti_sys_coef.info()
opti_sys_coef.head()

In [ ]:
# # creating derived Pi-Coefficients
# print("---- Creating Derived Pi-Coefficients ----")
# # derived coefficients columns
# # defining plot colums
# plot_cols = []

# # >>> working variables for derived a new coefficient
# # Service Effectiveness Coef: Pi_6 = Pi_2 * Pi_3^{-1} = chi / mu
# name = "Eff"
# print(f"Creating System Congestion Coefficient: \\Pi_{{{name}}}")

# # Pi_2 = mu / lambda
# temp = f"\\Pi_{{2}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
# pi_2 = opti_sys_coef[temp]

# # Pi_3 = chi / lambda
# temp = f"\\Pi_{{3}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
# pi_3 = opti_sys_coef[temp]

# numerator = "\\chi"
# denominator = "\\mu"

# # Pi_8 = chi * lambda / (mu * lambda) = chi / mu
# eff_coef = f"\\eta="
# eff_coef += f"\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
# opti_sys_coef[eff_coef] = pi_2 * (pi_3**-1)

# # >>> working variables for derived a new coefficient
# # Stall Coefficient: Pi_7 = P_1 = lambda * W / c
# name = "Stall"
# print(f"Creating System Stall Coefficient: \\Pi_{{{name}}}")

# # Pi_1 = lambda * W / c
# numerator = f"\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
# denominator = f"c_{{{sys_name}}}"
# temp = f"\\Pi_{{1}}=\\frac{{{numerator}}}{{{denominator}}}"
# pi_1 = opti_sys_coef[temp]

# # Pi_7 = Lambda * W / c -> Pi_7 = log(Pi_7 + 1)
# stall_coef = f"S=\\Pi_{{{name}}}=\\ln (\\frac{{\\lambda*W}}{{c}} + 1)"
# opti_sys_coef[stall_coef] = np.log(pi_1 + 1)

# # >>> working variables for derived a new coefficient
# # Occupation Coefficient: Pi_8 = Pi_5 * Pi_4^{-1} = L / K
# name = "Occ"
# print(f"Creating System Occupancy Coefficient: \\Pi_{{{name}}}")

# # Pi_4 = L / c
# temp = f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
# pi_4 = opti_sys_coef[temp]

# # Pi_5 = K / c
# temp = f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
# pi_5 = opti_sys_coef[temp]

# numerator = "L"
# denominator = "K"

# # Pi_8 = L * c / (K * c) = L / K
# occ_coef = f"O=\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
# opti_sys_coef[occ_coef] = pi_5 * (pi_4**-1)

# # change order to change chart distribution
# # x-axis, y-axis, z-axis/contourn
# plot_cols.append(occ_coef)
# plot_cols.append(stall_coef)
# plot_cols.append(eff_coef)

# # checking derived coefficients
# print(f"Derived Coefficients: {len(plot_cols)}, {plot_cols}")

# # sort by first colum from lower to highest value
# # opti_sys_coef.sort_values(by=plot_cols[0], inplace=True)
# # opti_sys_coef.reset_index(drop=True, inplace=True)

In [ ]:
# creating derived Pi-Coefficients
print("---- Creating Derived Pi-Coefficients ----")
# derived coefficients columns
# defining plot colums
plot_cols = []

# >>> working variables for derived a new coefficient
# Service Effectiveness Coef: Pi_6 = Pi_2 * Pi_3^{-1} = chi / mu
name = "Eff"
print(f"Creating System Congestion Coefficient: \\Pi_{{{name}}}")

# Pi_2 = mu / lambda
temp = f"\\Pi_{{2}}=\\frac{{\\mu_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_2 = opti_sys_coef[temp]

# Pi_3 = chi / lambda
temp = f"\\Pi_{{3}}=\\frac{{\\chi_{{{sys_name}}}}}{{\\lambda_{{{sys_name}}}}}"
pi_3 = opti_sys_coef[temp]

numerator = "\\chi"
denominator = "\\mu"

# Pi_8 = chi * lambda / (mu * lambda) = chi / mu
eff_coef = f"\\eta="
eff_coef += f"\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
opti_sys_coef[eff_coef] = pi_2 * (pi_3**-1)

# >>> working variables for derived a new coefficient
# normalizing P_1 by another coefficient to improve readbility
# Stall Coefficient: Pi_7 = P_1 * P_4^(-1) = lambda * W / K
# Adjusted in log-space: Ln(P_7 + 1 )= Ln(lambda * W / K + 1)
name = "Stall"
print(f"Creating System Stall Coefficient: \\Pi_{{{name}}}")

# Pi_1 = lambda * W / c
numerator = f"\\lambda_{{{sys_name}}}*W_{{{sys_name}}}"
denominator = f"c_{{{sys_name}}}"
temp = f"\\Pi_{{1}}=\\frac{{{numerator}}}{{{denominator}}}"
pi_1 = opti_sys_coef[temp]

# Pi_4 = K / c
temp = f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_4 = opti_sys_coef[temp]

# Pi_7 = Lambda * W / c -> Pi_7 = ln(Pi_7 + 1)
stall_coef = f"S=\\Pi_{{{name}}}=\\ln(\\frac{{\\lambda*W}}{{K}} + 1)"
pi_7 = pi_1 * (pi_4**-1)
opti_sys_coef[stall_coef] = np.log(pi_7 + 1)

# >>> working variables for derived a new coefficient
# Occupation Coefficient: Pi_8 = Pi_5 * Pi_4^{-1} = L / K
name = "Occ"
print(f"Creating System Occupancy Coefficient: \\Pi_{{{name}}}")

# Pi_4 = L / c
temp = f"\\Pi_{{4}}=\\frac{{K_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_4 = opti_sys_coef[temp]

# Pi_5 = L / c
temp = f"\\Pi_{{5}}=\\frac{{L_{{{sys_name}}}}}{{c_{{{sys_name}}}}}"
pi_5 = opti_sys_coef[temp]

numerator = "L"
denominator = "K"

# Pi_8 = L * c / (K * c) = L / K
occ_coef = f"O=\\Pi_{{{name}}}=\\frac{{{numerator}}}{{{denominator}}}"
opti_sys_coef[occ_coef] = pi_5 * (pi_4**-1)

# change order to change chart distribution
# x-axis, y-axis, z-axis/contourn
plot_cols.append(occ_coef)
plot_cols.append(stall_coef)
plot_cols.append(eff_coef)

# checking derived coefficients
print(f"Derived Coefficients: {len(plot_cols)}, {plot_cols}")

# sort by first colum from lower to highest value
# opti_sys_coef.sort_values(by=plot_cols[0], inplace=True)
# opti_sys_coef.reset_index(drop=True, inplace=True)

In [ ]:
print("\n--- System Pi-Coefficients from Monte Carlo Simulation ---")

# save data
# select result folder
file_path = os.path.join(PATH, results_folder, cs_folder, data_folder)
print(f"Data path: {file_path}")
save(file_path, "opti_dimensional_net_coeffs.csv", opti_pi_coefs)
opti_sys_coef.info()
opti_sys_coef.head()

In [ ]:
# create plot dataframe
plot_df = pd.DataFrame(opti_sys_coef[plot_cols])

In [ ]:
print("--- Charting Occupation and Congestion Coefficients ---")

metrics = plot_df.columns.tolist()
labels = [
    "Occupation",       # "Effectiveness",
    "Stall",            # "Stall",
    "Efficiency",       # "Occupation",
]
contour_name = plot_cols[-1]
print(f"Contour levels from: {contour_name}")
print(f"No. of contour lines {len(contour_vals)}")
plot_df.info()

In [ ]:
# clear plot data
# clear x-axis: occ <= 1
LIM_FLOW = 1.0
LIM_BUFFER = 1.0
print(f"cleaning invalid values in: {eff_coef}")
plot_df = plot_df[plot_df[eff_coef] <= LIM_FLOW]
print(f"Plot data with {eff_coef} < {LIM_FLOW}: {plot_df.shape}")

# # clear y-axis: stall <= K/c <= something when K is max and c is min
# print(f"cleaning invalid values in: {stall_coef}")
# plot_df = plot_df[plot_df[stall_coef] <= (K_MAX / C_MIN)]
# print(f"Plot data with {stall_coef} < {K_MAX}/{C_MIN}: {plot_df.shape}")

# clear contour/z-axis: eta <= 1
print(f"cleaning invalid values in: {occ_coef}")
plot_df = plot_df[plot_df[occ_coef] <= LIM_BUFFER]
print(f"Plot data with {occ_coef} < {LIM_BUFFER}: {plot_df.shape}")
plot_df.info()

In [ ]:
# sorting by Occ, them by Stall, then by Eff
print(f"Sorting plot data by: {plot_cols}")
plot_df.sort_values(by=plot_cols, inplace=True)
plot_df.reset_index(drop=True, inplace=True)
plot_df.info()

In [ ]:
# plotting the queue network dimensionless chart
# selecting images folder
file_path = os.path.join(PATH, results_folder, cs_folder, img_folder)
print(f"Data path: {file_path}")

# plot dimensionless system chart
title = "TAS Performance Chart: After Adaptation"
plot_performance_coef_chart(plot_df,
                            contour_name,
                            contour_vals,
                            metrics,
                            labels,
                            title,
                            file_path,
                            "opti_dimensional_perf_chart.png",
                            percentile={"x": [0.01, 0.99],
                                        "y": [0.01, 0.99]},
                            scale={"x": "log", "y": "log"},
                            limits={"x": [1e-5, 1.0],
                                    "y": [1.9e-2, 1.3e2]},)

In [ ]:
# plot this data in regular chart
plot_df.info()
print(plot_df.describe())

###### **Loading Dimensional Variables**

###### **Creating Dimensional Model**

###### **Calculating Pi-Coefficients**

###### **Running Sensitivity Analysis**

###### **Running Monte Carlo Simulation**

###### **Plotting Dimensional Model Chart**

In [ ]:
import numpy.ma as ma
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from math import factorial
import warnings

import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Visualization of Dimensionless Parameters and their relationships
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

In [ ]:
temp = pd.DataFrame(opti_sys_coef[plot_cols])
temp.info()

In [ ]:
print(eff_coef, ",", occ_coef)

In [ ]:
# clear plot data
# clear x-axis: occ <= 1
LIM_FLOW = 1.0
LIM_BUFFER = 1.0
print(f"cleaning invalid values in: {eff_coef}")
temp = temp[temp[eff_coef] <= LIM_FLOW]
print(f"Plot data with {eff_coef} < {LIM_FLOW}: {temp.shape}")

# clear contour/z-axis: eta <= 1
print(f"cleaning invalid values in: {occ_coef}")
temp = temp[temp[occ_coef] <= LIM_BUFFER]
print(f"Plot data with {occ_coef} < {LIM_BUFFER}: {temp.shape}")
temp_2 = pd.DataFrame(temp)
temp_2.info()

In [ ]:
temp_2

In [ ]:
# Create a copy of the dataframe to work with
plot_data = temp_2.copy()

# Rename columns for better readability in plots
plot_data.columns = ['Occupation', 'Stall', 'Efficiency']

# Filter out extreme values to focus on the main distribution
# Using percentiles to determine reasonable limits
percentile_low = 1
percentile_high = 99

for col in plot_data.columns:
    low_val = np.percentile(plot_data[col], percentile_low)
    high_val = np.percentile(plot_data[col], percentile_high)
    plot_data = plot_data[(plot_data[col] >= low_val) &
                          (plot_data[col] <= high_val)]

# Reset index after filtering
plot_data.reset_index(drop=True, inplace=True)

# Create figure with subplots
fig = plt.figure(figsize=(16, 10))

# Create 3D scatter plot
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
scatter = ax1.scatter(
    plot_data['Occupation'],
    plot_data['Stall'],
    plot_data['Efficiency'],
    c=plot_data['Efficiency'],
    cmap='viridis',
    alpha=0.5,
    s=10
)
ax1.set_xlabel('Occupation (L/K)')
ax1.set_ylabel('Stall (λW/c)')
ax1.set_zlabel('Efficiency (χ/μ)')
ax1.set_title('3D Visualization of TAS Dimensionless Parameters')
fig.colorbar(scatter, ax=ax1, label='Efficiency Value')

# Create pairwise scatter plots
ax2 = fig.add_subplot(2, 2, 2)
sns.scatterplot(data=plot_data, x='Occupation', y='Stall',
                hue='Efficiency', ax=ax2, palette='viridis', alpha=0.5, s=10)
ax2.set_title('Occupation vs. Stall')

ax3 = fig.add_subplot(2, 2, 3)
sns.scatterplot(data=plot_data, x='Occupation', y='Efficiency',
                hue='Stall', ax=ax3, palette='viridis', alpha=0.5, s=10)
ax3.set_title('Occupation vs. Efficiency')

ax4 = fig.add_subplot(2, 2, 4)
sns.scatterplot(data=plot_data, x='Stall', y='Efficiency',
                hue='Occupation', ax=ax4, palette='viridis', alpha=0.5, s=10)
ax4.set_title('Stall vs. Efficiency')

# Adjust layout
plt.tight_layout()
plt.suptitle(
    'TAS Performance Analysis: Relationships Between Dimensionless Parameters', fontsize=16)
plt.subplots_adjust(top=0.92)

# Display the plots
plt.show()

# Print summary statistics
print("Summary Statistics for Filtered Data:")
print(plot_data.describe())

In [ ]:
a =

##### **Base Configuration**

##### **Optimized Configuration**

#### **Data Analysis**

The steps are:

1) Extract and organize simulation data into a unique DataFrame.
2) Add metadata and basic statistics.
3) Save the dataframe to a CSV file.
4) Create a basic dimensionless plot (similar to Moody's chart)

##### **Data Post-Processing**


##### **Optimized Configuration**

#### **Numerical Solution (Simulation-Based)**

##### **Base Configuration**

##### **Optimized Configuration**

## **Results**

### **Compare Results**

### **Saving Results**

## **Analysis**

### **Graph Analysis**

## **Conclusion**

Understanding Contour Behavior in Your Queue System Chart
The contour lines in your plot represent queue length values (10, 20, 30, 40, 50), and they appear higher and closer to the Y-axis for higher values due to several important queue theory principles:

Why This Pattern Occurs
Exponential Queue Growth: In queueing theory, as system utilization (ρ = λ/μ) approaches 1.0, queue length grows exponentially rather than linearly. This fundamental property creates the compressed contour pattern you're seeing.

Dimensionless Variables Relationship: Your dimensionless plot shows that:

Small changes in the X-axis variable (first Pi coefficient) cause large changes in queue length when the system is near capacity
The Y-axis variable (second Pi coefficient) has less impact on queue length at higher utilization levels
Log-Log Scale Effect: You're using logarithmic scales on both axes, which compresses the higher values and spreads out the lower values, making this pattern more pronounced.

Queue Theory Interpretation
This pattern visualizes a critical queueing system concept: the performance cliff effect. As your system approaches saturation:

Small increases in load (moving right on X-axis) cause dramatically larger queue lengths
Improving service rate (moving up on Y-axis) provides diminishing returns once the system is near saturation
This is why the higher contour lines (40, 50) are compressed toward the right side of the chart and appear to "stack up" near the Y-axis.

Engineering Significance
This pattern in your data suggests:

Your system has a critical utilization threshold beyond which performance degrades rapidly
The "High Utilization Region" you've marked represents the danger zone where small load increases cause large queue growth
The system should be operated with sufficient margin from this threshold to maintain stable performance
This is exactly the kind of relationship a Moody-style chart should reveal - helping identify safe operating regions and performance boundaries in your system.

## **Future Work**

## **References & Sources**
<!-- TODO fix the references, links and details -->
1. [Queueing Theory](https://en.wikipedia.org/wiki/Queueing_theory)
2. [Dimensional Analysis](https://en.wikipedia.org/wiki/Dimensional_analysis)
3. [Simulation in Healthcare](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6466220/)

---

# **HASTA AKI!!!**